<a href="https://colab.research.google.com/github/MukeshRegar91/E-Commerce-Customer-Satisfaction-Score-Prediction/blob/main/The_project_of_flask_E_Commerce_Customer_Satisfaction_Score_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file == 'csat_model.h5':
            print("Model found:", os.path.join(root, file))

In [ ]:
import os

model_path = '/content/drive/MyDrive/csat_model.h5'

print("Model exists:", os.path.exists(model_path))

Model exists: False


In [ ]:
%pip -q install flask joblib

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
ARTIFACT_DIR = Path('/content/drive/MyDrive/CSAT_Deployment')
required = ['csat_model.keras', 'csat_scaler.joblib', 'csat_encoders.joblib', 'csat_feature_columns.joblib', 'example_request.json']
for name in required:
    print(f'{name}: {"FOUND" if (ARTIFACT_DIR / name).is_file() else "MISSING"}')
missing = [name for name in required if not (ARTIFACT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError('Missing artifacts: ' + ', '.join(missing) + '. Export these from FIRST notebook and rerun.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
csat_model.keras: FOUND
csat_scaler.joblib: FOUND
csat_encoders.joblib: FOUND
csat_feature_columns.joblib: FOUND
example_request.json: FOUND


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model

model = load_model(str(ARTIFACT_DIR / 'csat_model.keras'), compile=False)
scaler = joblib.load(ARTIFACT_DIR / 'csat_scaler.joblib')
encoders = joblib.load(ARTIFACT_DIR / 'csat_encoders.joblib')
feature_columns = list(joblib.load(ARTIFACT_DIR / 'csat_feature_columns.joblib'))
with open(ARTIFACT_DIR / 'example_request.json', encoding='utf-8') as f:
    example_request = json.load(f)

if model.input_shape[-1] != len(feature_columns):
    raise ValueError('ANN input size and feature columns do not match. Re-export matching training artifacts.')
if scaler.n_features_in_ != len(feature_columns):
    raise ValueError('Scaler and feature columns do not match. Re-export matching training artifacts.')
if not isinstance(encoders, dict):
    raise ValueError('Encoders must be a dictionary saved from the training notebook.')
print('Model and preprocessing loaded successfully!')
print('Number of features:', len(feature_columns))

Model and preprocessing loaded successfully!
Number of features: 13


In [ ]:
app = Flask(__name__)

def preprocess(payload):
    if not isinstance(payload, dict):
        raise ValueError('Send one JSON object with the required fields.')
    payload = dict(payload)

    # Match the training notebook: Response Time is in seconds.
    if 'Response Time' in feature_columns and 'Response Time' not in payload:
        if 'Issue_reported at' in payload and 'issue_responded' in payload:
            start = pd.to_datetime(payload['Issue_reported at'], dayfirst=True)
            finish = pd.to_datetime(payload['issue_responded'], dayfirst=True)
            payload['Response Time'] = (finish - start).total_seconds()

    missing = [col for col in feature_columns if col not in payload]
    if missing:
        raise ValueError('Missing input fields: ' + ', '.join(missing))

    row = {}
    for col in feature_columns:
        val = payload[col]
        if col in encoders:
            classes = encoders[col].classes_
            matches = [cl for cl in classes if str(cl) == str(val)]
            if not matches:
                raise ValueError(f'Unknown value for {col}: {val!r}. Use a training category.')
            row[col] = int(encoders[col].transform([matches[0]])[0])
        else:
            try:
                row[col] = float(val)
            except (TypeError, ValueError):
                raise ValueError(f'{col} must be numeric.')
            if not np.isfinite(row[col]):
                raise ValueError(f'{col} must be a finite number.')

    features = pd.DataFrame([row], columns=feature_columns)
    return scaler.transform(features)

@app.get('/')
def home():
    return jsonify(message='CSAT Prediction API is running', endpoints=['GET /example', 'POST /predict'])

@app.get('/example')
def example():
    return jsonify(example_request)

@app.post('/predict')
def predict():
    try:
        values = preprocess(request.get_json(silent=True))
        prediction = float(np.asarray(model.predict(values, verbose=0)).reshape(-1)[0])
        if not np.isfinite(prediction):
            raise ValueError('Model produced an invalid prediction.')
        return jsonify(predicted_CSAT_score=round(prediction, 3))
    except ValueError as err:
        return jsonify(error=str(err)), 400
    except Exception:
        app.logger.exception('Prediction failed')
        return jsonify(error='Internal prediction error'), 500

In [ ]:
# Test locally without public URL or running a background server
with app.test_client() as client:
    response = client.post('/predict', json=example_request)
    print('Homepage:', client.get('/').get_json())
    print('Status:', response.status_code)
    print('Prediction:', response.get_json())
    if response.status_code != 200:
        raise RuntimeError('Prediction test failed; see the error above.')
print('Flask prediction test PASSED')

Homepage: {'endpoints': ['GET /example', 'POST /predict'], 'message': 'CSAT Prediction API is running'}
Status: 200
Prediction: {'predicted_CSAT_score': 4.261}
Flask prediction test PASSED
